# RSNA Knee — CoAtNet Raptor only

元NotebookのCoAtNet Raptor primary checkpoint（`raptor_ft_coatnet_v5_full_swa.pt`）のみで`/kaggle/working/submission.csv`を生成します。旧checkpoint補完とTransformer系統との最終融合は除外しています。

## Stage 4 & 5: CoAtNet Raptor Arm & DINOsaur V4.2 Dual-Checkpoint Target Fusion (Score: 0.936)
- Evaluates `coatnet_rmlp_2_rw_384` hybrid on 64-slice multi-planar volume across 5 anatomical slots.
- Dual-GPU execution: Primary `coatnet_v5_swa` on GPU 0 ($w=0.89$, $K=64$) + complement `coatnet_v4` on GPU 1 ($w=0.11$, $K=42$).
- Stage 5 applies correlation-aware percentile rank fusion with proven target weights:
  `Lateral Meniscus: 0.61, Fracture: 0.61, Medial Meniscus: 0.56, Lateral OA: 0.55, ACL: 0.53`.


In [ ]:
# COATNET_TRANSFORMER_BLEND_V1
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
from concurrent.futures import ThreadPoolExecutor
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
IMG = 336
CROP_MM = 140.0
# 64 slices per study instead of 44, same proportions. Must match the corpus the weights
# were trained on (knee_corpus_v4.py).
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
         ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)                     # 64
K_EVAL = 62   # every window position the volume holds, not an evenly spaced subset
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [
    {
        "file": "raptor_ft_coatnet_v5_full_swa.pt",
        "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
        "res": 384,
        "w": 1.0,
    },
]

LEGACY_ARM = {
    "file": "raptor_ft_coatnet_v4_full.pt",
    "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
    "res": 384,
    "k_eval": 42,
    "span_lo": 0.06,
    "span_hi": 0.94,
}

PRIMARY_SPAN_LO = 0.02
PRIMARY_SPAN_HI = 0.98


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(
        device,
        non_blocking=True,
    )

    use_cuda = (
        str(device).startswith("cuda")
    )

    def _forward():
        return torch.sigmoid(
            model(x).float()
        )[0].cpu().numpy()

    if use_cuda:
        try:
            with torch.autocast(
                "cuda",
                dtype=torch.float16,
            ):
                return _forward()

        except RuntimeError as error:
            try:
                with torch.cuda.device(device):
                    torch.cuda.empty_cache()
            except Exception:
                pass

            print(
                "[DINOsaur V4.2] "
                f"{device} fp16 retry in fp32: "
                f"{type(error).__name__}",
                flush=True,
            )

            return _forward()

    return _forward()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def _fill_variant_volume(
    target_volume,
    offset,
    picks,
    pixel_cache,
    files,
    med_ps,
    mm_crop_resize,
):
    arrays = []
    spacings = []

    for position in picks:
        position = min(
            int(position),
            len(files) - 1,
        )
        file_path, spacing = files[
            position
        ]
        arrays.append(
            pixel_cache.get(
                position
            )
        )
        spacings.append(
            spacing
            if spacing > 0
            else med_ps
        )

    valid = [
        array
        for array in arrays
        if array is not None
    ]

    if valid:
        all_pixels = np.concatenate(
            [
                array.ravel()
                for array in valid
            ]
        )
        low, high = np.percentile(
            all_pixels,
            [2.0, 98.0],
        )
    else:
        low, high = 0.0, 1.0

    for local_index, (
        array,
        spacing,
    ) in enumerate(
        zip(
            arrays,
            spacings,
        )
    ):
        output_index = (
            offset
            + local_index
        )

        if (
            output_index
            >= MAXS
        ):
            break

        if array is None:
            continue

        normalized = np.clip(
            (
                array - low
            )
            / (
                high - low
                + 1e-6
            ),
            0,
            1,
        )

        normalized = mm_crop_resize(
            normalized,
            spacing,
        )

        target_volume[
            output_index
        ] = (
            normalized
            * 255
        ).astype(
            np.uint8
        )


def build_study_pair(
    sid,
    ser_records,
    tsdir,
    reader,
):
    """
    Produce exact MaxSpan and legacy-span volumes while reading every required
    DICOM only once. Both checkpoints keep their own percentile normalization.
    """
    (
        order_and_meta,
        read_px,
        mm_crop_resize,
    ) = reader

    rows = ser_records.get(
        sid,
        [],
    )

    primary_volume = np.zeros(
        (
            MAXS,
            IMG,
            IMG,
        ),
        np.uint8,
    )
    legacy_volume = np.zeros_like(
        primary_volume
    )

    used = set()
    offset = 0

    for plane, fluid, count in SLOTS:
        record = _pick_series_for_slot(
            rows,
            plane,
            fluid,
            used,
        )

        if record is None:
            offset += count
            continue

        used.add(
            record[
                "SeriesInstanceUID"
            ]
        )

        files, med_ps = order_and_meta(
            f"{tsdir}/{sid}/"
            f"{record['SeriesInstanceUID']}"
        )

        if not files:
            offset += count
            continue

        number = len(files)

        primary_low = int(
            number
            * PRIMARY_SPAN_LO
        )
        primary_high = int(
            number
            * PRIMARY_SPAN_HI
        ) - 1
        primary_high = max(
            primary_high,
            primary_low,
        )

        legacy_low = int(
            number
            * float(
                LEGACY_ARM[
                    "span_lo"
                ]
            )
        )
        legacy_high = int(
            number
            * float(
                LEGACY_ARM[
                    "span_hi"
                ]
            )
        ) - 1
        legacy_high = max(
            legacy_high,
            legacy_low,
        )

        if number > 1:
            primary_picks = np.linspace(
                primary_low,
                primary_high,
                count,
            ).round().astype(int)

            legacy_picks = np.linspace(
                legacy_low,
                legacy_high,
                count,
            ).round().astype(int)
        else:
            primary_picks = np.zeros(
                count,
                dtype=int,
            )
            legacy_picks = np.zeros(
                count,
                dtype=int,
            )

        required_positions = sorted(
            set(
                primary_picks.tolist()
                + legacy_picks.tolist()
            )
        )

        pixel_cache = {}

        for position in required_positions:
            position = min(
                int(position),
                number - 1,
            )

            file_path, _ = files[
                position
            ]

            try:
                pixel_cache[
                    position
                ] = read_px(
                    file_path
                )
            except Exception:
                pixel_cache[
                    position
                ] = None

        _fill_variant_volume(
            primary_volume,
            offset,
            primary_picks,
            pixel_cache,
            files,
            med_ps,
            mm_crop_resize,
        )

        _fill_variant_volume(
            legacy_volume,
            offset,
            legacy_picks,
            pixel_cache,
            files,
            med_ps,
            mm_crop_resize,
        )

        offset += count

        if offset >= MAXS:
            break

    primary_mask = (
        primary_volume.reshape(
            MAXS,
            -1,
        ).sum(1)
        > 0
    ).astype(
        np.uint8
    )

    legacy_mask = (
        legacy_volume.reshape(
            MAXS,
            -1,
        ).sum(1)
        > 0
    ).astype(
        np.uint8
    )

    return (
        primary_volume,
        primary_mask,
        legacy_volume,
        legacy_mask,
    )


# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(
    fname,
    required=True,
):
    direct = [
        f"/kaggle/input/raptor-knee-arms/{fname}",
        f"/kaggle/input/raptor-knee-arms/1/{fname}",
        f"/kaggle/input/raptor-cnn336/{fname}",
    ]

    for path in direct:
        if os.path.exists(path):
            return path

    for directory in sorted(
        glob.glob(
            "/kaggle/input/*/"
        )
    ):
        if (
            "competition"
            in directory.lower()
        ):
            continue

        hits = glob.glob(
            os.path.join(
                directory,
                "**",
                fname,
            ),
            recursive=True,
        )

        if hits:
            return hits[0]

    if required:
        raise RuntimeError(
            f"{fname} not found "
            "under /kaggle/input"
        )

    return None


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    reader = _make_reader()

    number_studies = len(
        test_ids
    )

    primary_predictions = np.full(
        (
            number_studies,
            len(LAB),
        ),
        0.5,
        np.float32,
    )

    legacy_predictions = np.full_like(
        primary_predictions,
        0.5,
    )

    legacy_success = np.zeros(
        number_studies,
        dtype=np.bool_,
    )

    if (
        torch.cuda.is_available()
        and torch.cuda.device_count() >= 1
    ):
        primary_device = torch.device(
            "cuda:0"
        )
    else:
        primary_device = torch.device(
            "cpu"
        )

    legacy_path = None
    legacy_enabled = False  # standalone primary checkpoint only

    primary_path = find_weight_file(
        ARMS[0][
            "file"
        ],
        required=True,
    )

    primary_model, primary_res = load_model(
        primary_path,
        ARMS[0][
            "arch"
        ],
        ARMS[0][
            "res"
        ],
        primary_device,
    )

    print(
        "[DINOsaur V4.2] primary "
        f"{ARMS[0]['file']} "
        f"on {primary_device}",
        flush=True,
    )

    legacy_model = None
    legacy_device = None
    legacy_res = None

    if legacy_enabled:
        legacy_device = torch.device(
            "cuda:1"
        )

        try:
            legacy_model, legacy_res = load_model(
                legacy_path,
                LEGACY_ARM[
                    "arch"
                ],
                LEGACY_ARM[
                    "res"
                ],
                legacy_device,
            )

            print(
                "[DINOsaur V4.2] complement "
                f"{LEGACY_ARM['file']} "
                f"on {legacy_device}",
                flush=True,
            )

        except Exception as error:
            legacy_enabled = False
            legacy_model = None

            print(
                "[DINOsaur V4.2] "
                "legacy checkpoint disabled "
                f"safely: "
                f"{type(error).__name__}: "
                f"{error}",
                flush=True,
            )

    else:
        print(
            "[DINOsaur V4.2] "
            "legacy complement unavailable "
            "or second GPU absent; "
            "exact 0.935 Raptor retained",
            flush=True,
        )

    executor = (
        ThreadPoolExecutor(
            max_workers=2
        )
        if legacy_enabled
        else None
    )

    for study_index, study_id in enumerate(
        test_ids
    ):
        try:
            (
                primary_volume,
                primary_mask,
                legacy_volume,
                legacy_mask,
            ) = build_study_pair(
                study_id,
                SER,
                tsdir,
                reader,
            )

            primary_windows = eval_windows(
                primary_volume,
                primary_mask,
                k=K_EVAL,
                res=primary_res,
                norm=NORM,
            )

            if legacy_enabled:
                legacy_windows = eval_windows(
                    legacy_volume,
                    legacy_mask,
                    k=int(
                        LEGACY_ARM[
                            "k_eval"
                        ]
                    ),
                    res=legacy_res,
                    norm=NORM,
                )

                primary_future = executor.submit(
                    infer_probs,
                    primary_model,
                    primary_windows,
                    primary_device,
                )

                legacy_future = executor.submit(
                    infer_probs,
                    legacy_model,
                    legacy_windows,
                    legacy_device,
                )

                primary_prediction = (
                    primary_future.result()
                )

                try:
                    legacy_prediction = (
                        legacy_future.result()
                    )
                    legacy_success[
                        study_index
                    ] = True
                except Exception as legacy_error:
                    legacy_prediction = (
                        primary_prediction.copy()
                    )

                    print(
                        "[DINOsaur V4.2] "
                        f"legacy study "
                        f"{study_index} fallback: "
                        f"{type(legacy_error).__name__}: "
                        f"{legacy_error}",
                        flush=True,
                    )

                del legacy_windows

            else:
                primary_prediction = infer_probs(
                    primary_model,
                    primary_windows,
                    primary_device,
                )
                legacy_prediction = (
                    primary_prediction.copy()
                )

            primary_predictions[
                study_index
            ] = primary_prediction

            legacy_predictions[
                study_index
            ] = legacy_prediction

            del (
                primary_volume,
                primary_mask,
                legacy_volume,
                legacy_mask,
                primary_windows,
                primary_prediction,
                legacy_prediction,
            )

        except Exception as error:
            print(
                "[DINOsaur V4.2] "
                f"study {study_index} "
                f"{study_id[:16]} FALLBACK "
                f"({type(error).__name__}: "
                f"{error})",
                flush=True,
            )

        if (
            (
                study_index + 1
            )
            % 100
            == 0
            or study_index + 1
            == number_studies
        ):
            print(
                "[DINOsaur V4.2] "
                f"{study_index+1}/"
                f"{number_studies} | "
                f"{time.time()-t0:.0f}s",
                flush=True,
            )

    if executor is not None:
        executor.shutdown(
            wait=True
        )

    del primary_model

    if legacy_model is not None:
        del legacy_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    primary_rank = rankpct(
        np.clip(
            primary_predictions,
            0,
            1,
        )
    )

    raptor_rank = (
        primary_rank.copy()
    )

    legacy_fraction = float(
        legacy_success.mean()
    ) if legacy_enabled else 0.0

    if (
        legacy_enabled
        and legacy_fraction >= 0.98
    ):
        legacy_rank = rankpct(
            np.clip(
                legacy_predictions,
                0,
                1,
            )
        )

        # The expanded MaxSpan checkpoint's documented largest gains are ACL,
        # both menisci, Lateral OA and Fracture. Keep it almost pure there.
        # Use the previous 0.934 checkpoint only for the remaining findings,
        # where it may restore complementary ordering.
        complement_weight = {
            "MCL": 0.16,
            "Medial OA": 0.10,
            "PF OA": 0.12,
            "Effusion": 0.10,
            "Synovitis": 0.16,
            "Baker's": 0.12,
            "Contusion": 0.12,
        }

        complement_log = []

        for target_index, target in enumerate(
            LAB
        ):
            weight = float(
                complement_weight.get(
                    target,
                    0.0,
                )
            )

            if weight <= 0:
                continue

            correlation = float(
                np.corrcoef(
                    primary_rank[
                        :,
                        target_index,
                    ],
                    legacy_rank[
                        :,
                        target_index,
                    ],
                )[0, 1]
            )

            if not np.isfinite(
                correlation
            ):
                weight = 0.0
            elif correlation > 0.992:
                weight *= 0.50
            elif correlation < 0.65:
                weight *= 0.40

            if weight <= 0:
                continue

            raptor_rank[
                :,
                target_index,
            ] = (
                (
                    1.0
                    - weight
                )
                * primary_rank[
                    :,
                    target_index,
                ]
                + weight
                * legacy_rank[
                    :,
                    target_index,
                ]
            )

            complement_log.append(
                (
                    target,
                    weight,
                    correlation,
                )
            )

        raptor_rank = rankpct(
            raptor_rank
        )

        print(
            "[DINOsaur V4.2] "
            "legacy complement: "
            + "; ".join(
                f"{target}=w{weight:.3f},"
                f"corr={correlation:.3f}"
                for (
                    target,
                    weight,
                    correlation,
                )
                in complement_log
            ),
            flush=True,
        )

    else:
        print(
            "[DINOsaur V4.2] "
            f"legacy success={legacy_fraction:.3f}; "
            "exact primary Raptor used",
            flush=True,
        )

    ranks = raptor_rank
    # ranks already contain the dual-checkpoint Raptor prediction.
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()



